<a href="https://colab.research.google.com/github/listenmusiceveryday/AIB6_Progress_Note/blob/main/v2-tongue-medgemma-zeroshot-baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👅 Tongue Health — MedGemma Zero-Shot Baseline

**Goal:** ส่ง Test Set ให้ MedGemma-4b-it predict แบบ zero-shot แล้วคำนวณ F1/Precision/Recall เทียบกับ ground truth

**Hardware:** T4 (15 GB VRAM) — Google Colab Free Tier

> ⚠️ ต้องขอ access MedGemma บน Hugging Face ก่อน: https://huggingface.co/google/medgemma-4b-it

## 0️⃣ Check GPU

In [1]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Mon May 25 00:27:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1️⃣ Install Packages

In [2]:
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q datasets scikit-learn Pillow tqdm pandas
!pip install -q iterative-stratification
print("✓ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
✓ Done


## 2️⃣ Hugging Face Login

> ต้องใช้ token ที่มีสิทธิ์ access `google/medgemma-4b-it`

In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3️⃣ Configuration

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # ── Data ──────────────────────────────────────────────────────────────
    DATA_DIR:  str = "/content/drive/MyDrive/kaggle_dataset/shezhenv3-txt/images"
    CSV_FILE:  str = "/content/drive/MyDrive/kaggle_dataset/shezhenv3-txt/tongue_real.csv"
    IMAGE_COL: str = "Image_Name"
    LABEL_COLS: List[str] = field(default_factory=lambda: [
        'Color_Pink',
        'Color_Red',
        'Coating_None',
        'Coating_White',
        'Coating_Yellow',
        'Texture_None',
        'Texture_Geographic',
        'Texture_Cracked',
        'Texture_Dentate'
    ])

    # ── Split ─────────────────────────────────────────────────────────────
    TEST_RATIO:  float = 0.10   # 10% เป็น test set
    RANDOM_SEED: int   = 42

    # ── Model ─────────────────────────────────────────────────────────────
    MODEL_ID: str  = "google/medgemma-4b-it"
    USE_4BIT: bool = True       # จำเป็นสำหรับ T4

    # ── Inference ─────────────────────────────────────────────────────────
    MAX_NEW_TOKENS: int = 256   # ให้ model ตอบ JSON สั้น ๆ
    TEMPERATURE:    float = 0.1 # ต่ำ = deterministic
    BATCH_SIZE:     int   = 1   # zero-shot ทีละรูป (safe บน T4)

    # ── Output ────────────────────────────────────────────────────────────
    OUTPUT_DIR: str = "/content/medgemma_zeroshot_output"

CFG = Config()
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"✓ Labels ({len(CFG.LABEL_COLS)}): {CFG.LABEL_COLS}")
print(f"✓ Model : {CFG.MODEL_ID}")
print(f"✓ Output: {CFG.OUTPUT_DIR}")

✓ Labels (9): ['Color_Pink', 'Color_Red', 'Coating_None', 'Coating_White', 'Coating_Yellow', 'Texture_None', 'Texture_Geographic', 'Texture_Cracked', 'Texture_Dentate']
✓ Model : google/medgemma-4b-it
✓ Output: /content/medgemma_zeroshot_output


## 4️⃣ Load Data & Split Test Set

> ใช้ Multilabel Stratified Split เพื่อให้ test set มี label distribution ตรงกับ dataset รวม

In [6]:
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

df = pd.read_csv(CFG.CSV_FILE)
print(f"Total samples : {len(df)}")
print(f"Columns       : {df.columns.tolist()}")
display(df.head(3))

# Label distribution
print("\nLabel distribution:")
for col in CFG.LABEL_COLS:
    n = df[col].sum()
    print(f"  {col:25s}: {n:4d} ({n/len(df)*100:.1f}%)")

Total samples : 3220
Columns       : ['Image_Name', 'Color_Pink', 'Color_Red', 'Coating_None', 'Coating_White', 'Coating_Yellow', 'Texture_None', 'Texture_Geographic', 'Texture_Cracked', 'Texture_Dentate']


,Image_Name,Color_Pink,Color_Red,Coating_None,Coating_White,Coating_Yellow,Texture_None,Texture_Geographic,Texture_Cracked,Texture_Dentate
0,10255.jpg,0,1,0,1,0,0,0,0,0
1,A (219).jpg,0,1,0,1,0,0,0,0,0
2,10028.jpg,0,0,0,1,0,0,0,0,0



Label distribution:
  Color_Pink               :  169 (5.2%)
  Color_Red                :  629 (19.5%)
  Coating_None             :  169 (5.2%)
  Coating_White            : 2167 (67.3%)
  Coating_Yellow           :  522 (16.2%)
  Texture_None             :  169 (5.2%)
  Texture_Geographic       :  201 (6.2%)
  Texture_Cracked          :  923 (28.7%)
  Texture_Dentate          :  631 (19.6%)


In [7]:
# Stratified split — แยก test 10% ออกมาก่อน (train/val ไม่ใช้ใน script นี้)
X = df[[CFG.IMAGE_COL]].values
y = df[CFG.LABEL_COLS].values

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=CFG.TEST_RATIO,
    random_state=CFG.RANDOM_SEED
)
train_val_idx, test_idx = next(msss.split(X, y))

test_df  = df.iloc[test_idx].reset_index(drop=True)
print(f"\n{'='*50}")
print(f"Test set size : {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"{'='*50}")

# ตรวจ label distribution ของ test set
print("\nTest label distribution:")
for col in CFG.LABEL_COLS:
    n = test_df[col].sum()
    print(f"  {col:25s}: {n:3d} ({n/len(test_df)*100:.1f}%)")


Test set size : 322 samples (10.0%)

Test label distribution:
  Color_Pink               :  17 (5.3%)
  Color_Red                :  63 (19.6%)
  Coating_None             :  17 (5.3%)
  Coating_White            : 217 (67.4%)
  Coating_Yellow           :  52 (16.1%)
  Texture_None             :  17 (5.3%)
  Texture_Geographic       :  20 (6.2%)
  Texture_Cracked          :  92 (28.6%)
  Texture_Dentate          :  63 (19.6%)


## 5️⃣ Load MedGemma-4b-it

> 4-bit quantization ให้ใช้ได้บน T4 (15 GB) — ใช้ VRAM ประมาณ 6-8 GB

In [8]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch

print(f"Loading {CFG.MODEL_ID} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if CFG.USE_4BIT else None

processor = AutoProcessor.from_pretrained(CFG.MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    CFG.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print(f"\n✓ Model loaded")
print(f"  Device     : {next(model.parameters()).device}")

# VRAM usage
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM used  : {used:.1f} / {total:.1f} GB")

Loading google/medgemma-4b-it ...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]


✓ Model loaded
  Device     : cuda:0
  VRAM used  : 3.2 / 15.6 GB


## 6️⃣ Prompt Engineering

> ออกแบบ prompt ให้ model ตอบ **JSON เท่านั้น** เพื่อให้ parse ง่าย

In [9]:
def build_prompt(label_cols):
    """
    สร้าง prompt ที่บอก model ชัดเจนว่าต้องการ JSON อะไรบ้าง
    """
    # แยก label เป็น group
    groups = {}
    for col in label_cols:
        prefix, val = col.split("_", 1)
        groups.setdefault(prefix, []).append(val)

    group_lines = "\n".join(
        f"- {grp}: options are {vals} (1 = present, 0 = absent)"
        for grp, vals in groups.items()
    )

    json_keys = "{" + ", ".join(f'"{c}": 0 or 1' for c in label_cols) + "}"

    prompt = f"""You are a medical AI assistant specializing in tongue diagnosis.
Examine the tongue image carefully and classify each feature below.

Features to classify:
{group_lines}

Rules:
- Color: EXACTLY ONE of Color_Pink or Color_Red must be 1
- Coating: EXACTLY ONE of Coating_None, Coating_White, Coating_Yellow must be 1
- Texture: EXACTLY ONE of Texture_None, Texture_Geographic, Texture_Cracked, Texture_Dentate must be 1
- All other values must be 0

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Format: {json_keys}"""
    return prompt

PROMPT = build_prompt(CFG.LABEL_COLS)
print("=== PROMPT ===")
print(PROMPT)

=== PROMPT ===
You are a medical AI assistant specializing in tongue diagnosis.
Examine the tongue image carefully and classify each feature below.

Features to classify:
- Color: options are ['Pink', 'Red'] (1 = present, 0 = absent)
- Coating: options are ['None', 'White', 'Yellow'] (1 = present, 0 = absent)
- Texture: options are ['None', 'Geographic', 'Cracked', 'Dentate'] (1 = present, 0 = absent)

Rules:
- Color: EXACTLY ONE of Color_Pink or Color_Red must be 1
- Coating: EXACTLY ONE of Coating_None, Coating_White, Coating_Yellow must be 1
- Texture: EXACTLY ONE of Texture_None, Texture_Geographic, Texture_Cracked, Texture_Dentate must be 1
- All other values must be 0

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Format: {"Color_Pink": 0 or 1, "Color_Red": 0 or 1, "Coating_None": 0 or 1, "Coating_White": 0 or 1, "Coating_Yellow": 0 or 1, "Texture_None": 0 or 1, "Texture_Geographic": 0 or 1, "Texture_Cracked": 0 or 1, "Texture_Dentate": 0 or 1

## 7️⃣ Zero-Shot Inference

> วนส่งรูปทีละใบ → parse JSON → เก็บผล

In [10]:
import json, re
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

def parse_json_response(text: str, label_cols: list) -> dict:
    """
    พยายาม parse JSON จาก response ของ model
    ถ้า parse ไม่ได้ → return None label ทั้งหมดเป็น -1 (unknown)
    """
    # ลอง extract JSON จาก text
    try:
        # กรณีมี markdown code block
        match = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        # กรณี JSON ล้วน
        match = re.search(r'{[^{}]+}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return None

def response_to_labels(parsed: dict, label_cols: list) -> list:
    """
    แปลง parsed dict → list of int ตาม label_cols order
    ค่าที่ไม่ได้รับ หรือ parse ไม่ได้ → -1 (จะถูก exclude ตอนคำนวณ metric)
    """
    if parsed is None:
        return [-1] * len(label_cols)
    result = []
    for col in label_cols:
        val = parsed.get(col, -1)
        try:
            result.append(int(val))
        except (ValueError, TypeError):
            result.append(-1)
    return result

@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    """Predict labels for a single image"""
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            temperature=cfg.TEMPERATURE,
            do_sample=False,   # greedy สม่ำเสมอกว่าสำหรับ eval
        )

        # decode เฉพาะส่วนที่ generate ใหม่
        input_len = inputs["input_ids"].shape[-1]
        generated = outputs[0][input_len:]
        response_text = processor.decode(generated, skip_special_tokens=True)
        return response_text, None

    except Exception as e:
        return None, str(e)

print("✓ Inference functions ready")

✓ Inference functions ready


In [43]:
# ── Run Inference ────────────────────────────────────────────────────────
results = []
errors  = []

data_dir = Path(CFG.DATA_DIR)

print(f"Running zero-shot inference on {len(test_df)} images ...")
print(f"Model: {CFG.MODEL_ID}\n")

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
    img_path = data_dir / row[CFG.IMAGE_COL]

    raw_text, error = predict_single(img_path, PROMPT, model, processor, CFG)

    if error:
        errors.append({"idx": i, "image": row[CFG.IMAGE_COL], "error": error})
        pred_labels = [-1] * len(CFG.LABEL_COLS)
        raw_text = ""
    else:
        parsed = parse_json_response(raw_text, CFG.LABEL_COLS)
        pred_labels = response_to_labels(parsed, CFG.LABEL_COLS)

    results.append({
        "image":      row[CFG.IMAGE_COL],
        "raw_output": raw_text,
        "pred":       pred_labels,
        "true":       [int(row[c]) for c in CFG.LABEL_COLS],
    })

print(f"\n✓ Done — {len(results)} images processed, {len(errors)} errors")
if errors:
    print(f"\n⚠️  Errors:")
    for e in errors[:5]:
        print(f"   {e['image']}: {e['error']}")

Running zero-shot inference on 322 images ...
Model: google/medgemma-4b-it



Predicting:   0%|          | 0/322 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_6842/2898769906.py", line 13, in <cell line: 0>
    raw_text, error = predict_single(img_path, PROMPT, model, processor, CFG)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6842/2097675740.py", line 29, in predict_single
    outputs = model.generate(
              ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2580, in gene

TypeError: object of type 'NoneType' has no len()

In [36]:
# ทดสอบ 1 รูป
row = test_df.iloc[0]
img_path = Path(CFG.DATA_DIR) / row[CFG.IMAGE_COL]

raw_text, error = predict_single(img_path, PROMPT, model, processor, CFG)
print(f"Error : {error}")
print(f"Raw   : {raw_text[-300:]}")  # ดู 300 ตัวท้าย

parsed = parse_json_response(raw_text, CFG.LABEL_COLS)
print(f"\nParsed: {parsed}")

pred = response_to_labels(parsed, CFG.LABEL_COLS)
true = [int(row[c]) for c in CFG.LABEL_COLS]
print(f"\nPred : {dict(zip(CFG.LABEL_COLS, pred))}")
print(f"True : {dict(zip(CFG.LABEL_COLS, true))}")

Error : None
Raw   : 

Parsed: None

Pred : {'Color_Pink': -1, 'Color_Red': -1, 'Coating_None': -1, 'Coating_White': -1, 'Coating_Yellow': -1, 'Texture_None': -1, 'Texture_Geographic': -1, 'Texture_Cracked': -1, 'Texture_Dentate': -1}
True : {'Color_Pink': 0, 'Color_Red': 0, 'Coating_None': 0, 'Coating_White': 1, 'Coating_Yellow': 0, 'Texture_None': 0, 'Texture_Geographic': 0, 'Texture_Cracked': 0, 'Texture_Dentate': 0}


In [37]:
# ทดสอบ text-only ก่อน — ไม่มีรูปเลย
text_inputs = processor.tokenizer(
    "What is 1+1? Answer in one word.",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    text_outputs = model.generate(
        **text_inputs,
        max_new_tokens=20,
        do_sample=False,
    )

response = processor.tokenizer.decode(
    text_outputs[0][text_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(f"Text-only response: [{response}]")

Text-only response: []


In [38]:
# โหลดใหม่ด้วย 8-bit
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    CFG.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

# ทดสอบ text-only ใหม่
text_inputs = processor.tokenizer(
    "What is 1+1? Answer in one word.",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    text_outputs = model.generate(
        **text_inputs,
        max_new_tokens=20,
        do_sample=False,
    )

response = processor.tokenizer.decode(
    text_outputs[0][text_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(f"Text-only response: [{response}]")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Text-only response: [

What is 1+1?

What is 1+1?

What is ]


In [39]:
# ทดสอบ text-only พร้อม generation params ที่ดีขึ้น
with torch.no_grad():
    text_outputs = model.generate(
        **text_inputs,
        max_new_tokens=20,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

response = processor.tokenizer.decode(
    text_outputs[0][text_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(f"Text-only response: [{response}]")

Text-only response: [

Submitted by Sarah J. Mar. 23, 2024 05]


In [40]:
# ลบ model เก่าออกก่อน เพื่อเคลียร์ VRAM
del model
torch.cuda.empty_cache()
import gc; gc.collect()

model = AutoModelForImageTextToText.from_pretrained(
    CFG.MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,   # ✅ native dtype ของ Gemma3
)
model.eval()

# ทดสอบ text-only
text_inputs = processor.tokenizer(
    "What is 1+1? Answer in one word.",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    text_outputs = model.generate(
        **text_inputs,
        max_new_tokens=20,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

response = processor.tokenizer.decode(
    text_outputs[0][text_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(f"Text-only response: [{response}]")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Text-only response: [

Submitted by Sarah M. Mar. 23, 2024 05]


In [41]:
# ใช้ chat template แม้แต่ text-only
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What is 1+1? Answer in one word."}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

response = processor.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)
print(f"Response: [{response}]")

Response: [Two
]


In [42]:
row = test_df.iloc[0]
img_path = Path(CFG.DATA_DIR) / row[CFG.IMAGE_COL]
image = Image.open(img_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=CFG.MAX_NEW_TOKENS,
        do_sample=False,
    )

response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
print(f"Response: [{response}]")
print(f"\nTrue: {dict(zip(CFG.LABEL_COLS, [int(row[c]) for c in CFG.LABEL_COLS]))}")

Response: [```json
{"Color_Pink": 1, "Color_Red": 0, "Coating_None": 0, "Coating_White": 0, "Coating_Yellow": 0, "Texture_None": 0, "Texture_Geographic": 0, "Texture_Cracked": 0, "Texture_Dentate": 0}
```]

True: {'Color_Pink': 0, 'Color_Red': 0, 'Coating_None': 0, 'Coating_White': 1, 'Coating_Yellow': 0, 'Texture_None': 0, 'Texture_Geographic': 0, 'Texture_Cracked': 0, 'Texture_Dentate': 0}


In [43]:
@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": prompt},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)  # ✅ ไม่ใส่ dtype

        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        response = processor.decode(
            outputs[0][input_len:],
            skip_special_tokens=True
        )
        return response, None

    except Exception as e:
        return None, str(e)

print("✓ Final version — ready to run inference loop")

✓ Final version — ready to run inference loop


In [34]:
# ดู raw tokens ดิบๆ ก่อนเลย
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "What color is this tongue? Answer in one word."},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, torch.float16)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
    )

full = processor.decode(outputs[0], skip_special_tokens=False)
print(repr(full))  # ใช้ repr เพื่อเห็น special tokens ทุกตัว

'<bos><start_of_turn>user\n\n\n<start_of_image><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token><image_soft_token

In [35]:
@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": prompt},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)  # ✅ ไม่ใส่ dtype

        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        # decode เฉพาะส่วนที่ generate ใหม่
        generated_tokens = outputs[0][input_len:]
        response_text = processor.decode(generated_tokens, skip_special_tokens=True)
        return response_text, None

    except Exception as e:
        return None, str(e)

print("✓ Updated")

✓ Updated


In [31]:
# Debug token lengths
row = test_df.iloc[0]
img_path = Path(CFG.DATA_DIR) / row[CFG.IMAGE_COL]
image = Image.open(img_path).convert("RGB")

image_token = processor.boi_token
text_with_image_token = image_token + "\n" + PROMPT

inputs = processor(
    images=image,
    text=text_with_image_token,
    return_tensors="pt"
).to(model.device, torch.float16)

input_len = inputs["input_ids"].shape[-1]
print(f"Input tokens: {input_len}")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=CFG.MAX_NEW_TOKENS,
        do_sample=False,
    )

output_len = outputs.shape[-1]
print(f"Output tokens: {output_len}")
print(f"New tokens generated: {output_len - input_len}")

# decode แบบต่างๆ
print("\n=== Full decode (ท้าย 500 ตัว) ===")
print(processor.decode(outputs[0], skip_special_tokens=False)[-500:])

print("\n=== Sliced decode ===")
print(f"[{processor.decode(outputs[0][input_len:], skip_special_tokens=True)}]")

Input tokens: 579
Output tokens: 835
New tokens generated: 256

=== Full decode (ท้าย 500 ตัว) ===
<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

=== Sliced decode ===
[]


In [32]:
@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": prompt},
                ],
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device, torch.float16)

        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        # ✅ decode full แล้วใช้ split("<end_of_turn>") เอาแค่ส่วน response
        full = processor.decode(outputs[0], skip_special_tokens=False)

        # Gemma3 ใช้ <end_of_turn> คั่นระหว่าง user/model turn
        if "<end_of_turn>" in full:
            parts = full.split("<end_of_turn>")
            # part สุดท้ายคือ response ของ model
            response = parts[-1].strip()
        else:
            # fallback: เอา 500 ตัวท้าย
            response = full[-500:].strip()

        # ลบ special tokens ที่เหลือ
        for tok in ["<bos>", "<eos>", "<pad>", "<start_of_turn>", "model", "user"]:
            response = response.replace(tok, "")
        response = response.strip()

        return response, None

    except Exception as e:
        return None, str(e)

print("✓ Updated")

✓ Updated


In [29]:
@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    try:
        # ✅ ใช้ boi_token ที่ถูกต้องของ Gemma3
        image_token = processor.boi_token  # <start_of_image>
        text_with_image_token = image_token + "\n" + prompt

        inputs = processor(
            images=image,
            text=text_with_image_token,
            return_tensors="pt"
        ).to(model.device, torch.float16)

        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        generated_tokens = outputs[0][input_len:]
        response_text = processor.decode(generated_tokens, skip_special_tokens=True)
        return response_text, None

    except Exception as e:
        return None, str(e)

print("✓ Updated")

✓ Updated


In [28]:
# เช็คว่า image token ที่ถูกต้องคืออะไร
print(type(processor))
print(dir(processor))
print()

# ดู special tokens
if hasattr(processor, 'tokenizer'):
    print("Special tokens:", processor.tokenizer.special_tokens_map)
    print("Image token:", getattr(processor.tokenizer, 'image_token', 'NOT FOUND'))
    print("Boi token:", getattr(processor.tokenizer, 'boi_token', 'NOT FOUND'))

<class 'transformers.models.gemma3.processing_gemma3.Gemma3Processor'>
['__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_auto_class', '_check_special_mm_tokens', '_get_arguments_from_pretrained', '_get_files_timestamps', '_get_num_multimodal_tokens', '_load_tokenizer_from_pretrained', '_merge_kwargs', '_upload_modified_files', 'apply_chat_template', 'audio_ids', 'batch_decode', 'boi_token', 'chat_template', 'check_argument_for_proper_class', 'create_mm_token_type_ids', 'decode', 'from_args_and_dict', 'from_pretrained', 'full_image_sequence', 'get_attributes', 'get_possibly_dynamic_module', 'get_processor_dict', 'image_ids', 'image_processor', 'imag

## 8️⃣ Compute Metrics

In [12]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    hamming_loss, classification_report
)

# แยก valid predictions (ไม่มี -1) ออกมาคำนวณ
valid_preds  = []
valid_trues  = []
skipped      = 0

for r in results:
    if -1 in r["pred"]:
        skipped += 1
        continue
    valid_preds.append(r["pred"])
    valid_trues.append(r["true"])

y_pred = np.array(valid_preds)
y_true = np.array(valid_trues)

print(f"Valid predictions : {len(y_pred)} / {len(results)}")
print(f"Skipped (parse fail / error): {skipped}")
print()

if len(y_pred) == 0:
    print("❌ ไม่มี valid predictions เลย — ตรวจสอบ prompt หรือ model output")
else:
    metrics = {
        "f1_micro"    : f1_score(y_true, y_pred, average="micro",    zero_division=0),
        "f1_macro"    : f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_samples"  : f1_score(y_true, y_pred, average="samples",  zero_division=0),
        "precision"   : precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall"      : recall_score(y_true, y_pred, average="micro",    zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match" : float((y_pred == y_true).all(axis=1).mean()),
        "n_valid"     : len(y_pred),
        "n_total"     : len(results),
        "n_skipped"   : skipped,
    }

    print("=" * 55)
    print("ZERO-SHOT RESULTS  (MedGemma-4b-it)")
    print("=" * 55)
    print(f"  F1 Micro      : {metrics['f1_micro']:.4f}")
    print(f"  F1 Macro      : {metrics['f1_macro']:.4f}")
    print(f"  F1 Samples    : {metrics['f1_samples']:.4f}")
    print(f"  Precision     : {metrics['precision']:.4f}")
    print(f"  Recall        : {metrics['recall']:.4f}")
    print(f"  Hamming Loss  : {metrics['hamming_loss']:.4f}")
    print(f"  Exact Match   : {metrics['exact_match']:.4f}")
    print("=" * 55)

Valid predictions : 0 / 322
Skipped (parse fail / error): 322

❌ ไม่มี valid predictions เลย — ตรวจสอบ prompt หรือ model output


In [14]:
# ดู raw output 5 อันแรก
for r in results[:5]:
    print(f"Image : {r['image']}")
    print(f"Output: [{r['raw_output']}]")
    print("-" * 60)

Image : 10258.jpg
Output: []
------------------------------------------------------------
Image : A (158).jpg
Output: []
------------------------------------------------------------
Image : 10404.jpg
Output: []
------------------------------------------------------------
Image : 10493.jpg
Output: []
------------------------------------------------------------
Image : A (153).jpg
Output: []
------------------------------------------------------------


In [15]:
# ทดสอบรูปเดียวแล้วดู raw output ก่อน decode
from PIL import Image
import torch

row = test_df.iloc[0]
img_path = Path(CFG.DATA_DIR) / row[CFG.IMAGE_COL]
image = Image.open(img_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

print(f"Input token length: {inputs['input_ids'].shape[-1]}")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=CFG.MAX_NEW_TOKENS,
        do_sample=False,
    )

print(f"Output token length: {outputs.shape[-1]}")

# ลอง decode แบบต่างๆ
full_text = processor.decode(outputs[0], skip_special_tokens=True)
print(f"\n=== FULL decode ===\n{full_text[-500:]}")  # 500 ตัวท้าย

input_len = inputs["input_ids"].shape[-1]
sliced = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
print(f"\n=== Sliced decode ===\n[{sliced}]")

Input token length: 586
Output token length: 842

=== FULL decode ===
e, Coating_White, Coating_Yellow must be 1
- Texture: EXACTLY ONE of Texture_None, Texture_Geographic, Texture_Cracked, Texture_Dentate must be 1
- All other values must be 0

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Format: {"Color_Pink": 0 or 1, "Color_Red": 0 or 1, "Coating_None": 0 or 1, "Coating_White": 0 or 1, "Coating_Yellow": 0 or 1, "Texture_None": 0 or 1, "Texture_Geographic": 0 or 1, "Texture_Cracked": 0 or 1, "Texture_Dentate": 0 or 1}
model


=== Sliced decode ===
[]


In [17]:
@torch.no_grad()
def predict_single(image_path, prompt, model, processor, cfg):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return None, f"Image load error: {e}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.MAX_NEW_TOKENS,
            do_sample=False,
        )

        # ✅ decode ทั้งหมด แล้วตัด prompt ออกด้วย string แทน token slice
        full_text = processor.decode(outputs[0], skip_special_tokens=True)

        # หา JSON จาก full_text โดยตรง (ไม่ต้อง slice token)
        return full_text, None

    except Exception as e:
        return None, str(e)

# ✅ แก้ parse_json_response ให้ดึง JSON จาก full text ได้
def parse_json_response(text: str, label_cols: list) -> dict:
    if not text:
        return None
    try:
        # หา { ... } อันสุดท้ายใน text (เป็น response ของ model)
        matches = list(re.finditer(r'\{[^{}]+\}', text, re.DOTALL))
        if matches:
            last_match = matches[-1]  # เอาอันสุดท้าย = ส่วนที่ model ตอบ
            return json.loads(last_match.group(0))
    except Exception:
        pass
    return None

print("✓ Updated — รัน inference loop ใหม่ได้เลย")

✓ Updated — รัน inference loop ใหม่ได้เลย


In [13]:
# Per-label breakdown
print("\nPer-Label F1 Score:")
print("-" * 45)
per_label_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
for label, score in zip(CFG.LABEL_COLS, per_label_f1):
    bar = "█" * int(score * 20)
    print(f"  {label:25s}: {score:.4f}  {bar}")

print()
print("Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=CFG.LABEL_COLS, zero_division=0))


Per-Label F1 Score:
---------------------------------------------

Detailed Classification Report:


ValueError: Number of classes, 0, does not match size of target_names, 9. Try specifying the labels parameter

## 9️⃣ Save Results

In [ ]:
import json

# Save raw results (รูป + prediction + ground truth)
results_path = f"{CFG.OUTPUT_DIR}/zeroshot_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

# Save metrics summary
metrics_path = f"{CFG.OUTPUT_DIR}/zeroshot_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

# Save predictions CSV (ง่ายต่อการ compare กับ fine-tuned model ทีหลัง)
pred_cols = {f"pred_{c}": [r["pred"][i] for r in results] for i, c in enumerate(CFG.LABEL_COLS)}
true_cols = {f"true_{c}": [r["true"][i] for r in results] for i, c in enumerate(CFG.LABEL_COLS)}
pred_df = pd.DataFrame({"image": [r["image"] for r in results], **pred_cols, **true_cols})
csv_path = f"{CFG.OUTPUT_DIR}/zeroshot_predictions.csv"
pred_df.to_csv(csv_path, index=False)

print(f"✓ Saved:")
print(f"  {results_path}")
print(f"  {metrics_path}")
print(f"  {csv_path}")

## 🔍 Inspect Sample Outputs

> ดู raw output ของ model เพื่อ debug prompt ถ้า parse rate ต่ำ

In [ ]:
print("Sample raw outputs (first 5):")
print("=" * 70)
for r in results[:5]:
    print(f"Image : {r['image']}")
    print(f"True  : {dict(zip(CFG.LABEL_COLS, r['true']))}")
    print(f"Pred  : {dict(zip(CFG.LABEL_COLS, r['pred']))}")
    print(f"Output: {r['raw_output'][:200]}")
    print("-" * 70)